In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from scipy import stats

In [ ]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, ROOT)

from src.split_data import split_data

cancerous = {"BCC", "SCC", "MEL"}

def make_features_labels(df, extra_drops=()):
    drops = ["img_id", "diagnostic", "Unnamed: 0", *extra_drops]
    features = df.drop(columns=drops, errors="ignore").select_dtypes(include="number")
    labels = np.where(df["diagnostic"].isin(cancerous), "cancerous", "non-cancerous")
    return features, labels

def make_forest(seed=6, n_estimators=200, max_depth=None):
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight="balanced",
        random_state=seed,
    )

In [ ]:
features_path = os.path.join(ROOT, "data", "features.csv")
splits_dir    = os.path.join(ROOT, "data", "splits")
train_df, val_df, test_df = split_data(features_path, 0.65, 0.20, 42, splits_dir)

_, train_labels = make_features_labels(train_df)
_, val_labels   = make_features_labels(val_df)
_, test_labels  = make_features_labels(test_df)

contrast_cols = ["contrast_diff", "contrast_ratio", "contrast_standardized"]
scenarios = [
    ["none",                  contrast_cols],
    ["contrast_diff",         ["contrast_ratio", "contrast_standardized"]],
    ["contrast_ratio",        ["contrast_diff", "contrast_standardized"]],
    ["contrast_standardized", ["contrast_diff", "contrast_ratio"]],
    ["all",                   []],
]

In [ ]:
results = {}
for name, drops in scenarios:
    train_X, _ = make_features_labels(train_df, drops)
    val_X,   _ = make_features_labels(val_df,   drops)
    test_X,  _ = make_features_labels(test_df,  drops)

    forest = make_forest()
    forest.fit(train_X, train_labels)

    val_preds  = forest.predict(val_X)
    test_preds = forest.predict(test_X)
    val_acc    = forest.score(val_X, val_labels)
    test_acc   = forest.score(test_X, test_labels)

    results[name] = dict(val_preds=val_preds, test_preds=test_preds,
                         val_acc=val_acc,     test_acc=test_acc)
    print(f"{name:25s}  val={val_acc:.3f}  test={test_acc:.3f}")

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(9, 22))
for i, (name, r) in enumerate(results.items()):
    ConfusionMatrixDisplay.from_predictions(val_labels,  r["val_preds"],  cmap="Blues", ax=axes[i, 0], colorbar=False)
    ConfusionMatrixDisplay.from_predictions(test_labels, r["test_preds"], cmap="Blues", ax=axes[i, 1], colorbar=False)
    axes[i, 0].set_title(f"{name} — val (acc={r['val_acc']:.3f})")
    axes[i, 1].set_title(f"{name} — test (acc={r['test_acc']:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
N_SEEDS = 200
importances = {name: [] for name, _ in scenarios}
val_accs    = {name: [] for name, _ in scenarios}
test_accs   = {name: [] for name, _ in scenarios}

for seed in range(N_SEEDS):
    for name, drops in scenarios:
        train_X, _ = make_features_labels(train_df, drops)
        val_X,   _ = make_features_labels(val_df,   drops)
        test_X,  _ = make_features_labels(test_df,  drops)

        forest = make_forest(seed)
        forest.fit(train_X, train_labels)

        importances[name].append(pd.Series(forest.feature_importances_, index=train_X.columns))
        val_accs[name].append(forest.score(val_X, val_labels))
        test_accs[name].append(forest.score(test_X, test_labels))
    print(f"seed {seed+1}/{N_SEEDS} done")

importance_dfs = {name: pd.DataFrame(rows) for name, rows in importances.items()}
val_acc_df  = pd.DataFrame(val_accs)
test_acc_df = pd.DataFrame(test_accs)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 28))
for ax, (name, imp_df) in zip(axes, importance_dfs.items()):
    means = imp_df.mean().sort_values()
    stds  = imp_df.std().reindex(means.index)
    means.plot.barh(ax=ax, xerr=stds, color="steelblue", ecolor="black")
    ax.set_title(f"{name} — mean Gini importance ± std over {N_SEEDS} seeds")
    ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
test_acc_df.boxplot(ax=ax)
ax.set_ylabel("test accuracy")
ax.set_title(f"Test accuracy distribution over {N_SEEDS} seeds")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# paired Wilcoxon vs. the 'all' baseline
print(f"Paired Wilcoxon signed-rank test on test accuracy (vs. 'all'), n={N_SEEDS}:")
print(f"  {'scenario':25s}  {'mean acc':>10s}  {'mean diff':>10s}  {'p-value':>10s}")
for name, _ in scenarios:
    mean_acc = test_acc_df[name].mean()
    if name == "all":
        print(f"  {name:25s}  {mean_acc:>10.4f}  {'(baseline)':>10s}  {'':>10s}")
        continue
    diffs = test_acc_df["all"] - test_acc_df[name]
    stat, p = stats.wilcoxon(test_acc_df["all"], test_acc_df[name], zero_method="wilcox")
    print(f"  {name:25s}  {mean_acc:>10.4f}  {diffs.mean():>+10.4f}  {p:>10.10f}")